# CERES-EMIT-MODIS Collocation

In [1]:
import earthaccess
import xarray as xr
#from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import h5netcdf
import awscli, s3fs, boto3
import requests
import sys, os, re, glob,datetime,pickle


/home/ec2-user/Softwares/anaconda3/envs/PACE_EMIT/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os,sys
sys.path.append("/home/ec2-user/Research/python_lib")
import AWS_Utils
from AWS_Utils import get_NASA_creds
import EMIT_lib
from haversine import haversine

## Set up AWS credentials and bucket 

In [3]:
lpdaac_creds = get_NASA_creds('lpdaac')
fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=lpdaac_creds['accessKeyId'], 
                          secret=lpdaac_creds['secretAccessKey'], 
                          token=lpdaac_creds['sessionToken'])

In [4]:
# my_bucket = "s3://zhibo-zhang-bucket"
# files = fs_s3.ls(my_bucket)[-3:]

In [5]:
auth = earthaccess.login(persist=True,strategy="netrc")

## use NASA Earthaccess to search for CERES, EMIT and MODIS data


### set up the domain for the study

In [6]:
domain_lat_min, domain_lat_max = 14.3,34.7
domain_lon_min, domain_lon_max = -17.3,60.2
domain_box = (  domain_lon_min, domain_lat_min,
                domain_lon_max, domain_lat_max )
import datetime
study_period_start = datetime.date(2023,6,1)
study_period_end = datetime.date(2023,8,31)
# Format dates as YYYY-MM-DD
formatted_start = study_period_start.strftime('%Y-%m-%d')
formatted_end = study_period_end.strftime('%Y-%m-%d')

print(formatted_start)  # Output: 2023-06-01
print(formatted_end)    # Output: 2023-08-31

2023-06-01
2023-08-31


### first search for CERES products

In [7]:
SSF_Aqua_FM3_products = earthaccess.search_data(
     short_name="CER_SSF_Aqua-FM3-MODIS",
     bounding_box=domain_box,
     temporal = (formatted_start,formatted_end)
    )
print('number of CERES SSF files found', len(SSF_Aqua_FM3_products))

number of CERES SSF files found 1548


In [9]:
CERES_SSF_directory = "./CERES_SSF/"
for i in range(len(SSF_Aqua_FM3_products)):
    filename = os.path.basename(SSF_Aqua_FM3_products[i].data_links()[0])
    #print(i)
    if not filename.endswith(".hdf"):
        filepath = os.path.join(CERES_SSF_directory, filename+".hdf")
    else:
        filepath = os.path.join(CERES_SSF_directory, filename)
    #print(filepath)
    if not os.path.exists(filepath):
        print(i,filename,"File does not exist...download it")
        #earthaccess.download(SSF_Aqua_FM3_products[i], "./CERES_SSF")
        
    # else:
    #     print(filename,"File does exist")

### download CERES data

In [11]:
earthaccess.download(SSF_Aqua_FM3_products[0])

QUEUEING TASKS | : 100%|██████████| 1/1 [00:00<00:00, 646.17it/s]
PROCESSING TASKS | : 100%|██████████| 1/1 [00:56<00:00, 56.21s/it]
COLLECTING RESULTS | : 100%|██████████| 1/1 [00:00<00:00, 22550.02it/s]


['/home/ec2-user/Research/EMIT_dust_DRE/data/2025-03-23-4a1caf/CER_SSF_Aqua-FM3-MODIS_Edition4A_407405.2023060100.hdf']

In [15]:
#earthaccess.download(SSF_Aqua_FM3_products, "./CERES_SSF")

### search for EMIT data

In [16]:
EMIT_products = earthaccess.search_datasets(
     project="EMIT",
     # bounding_box=domain_box,
     # temporal = (formatted_start,formatted_end)
    )
for item in EMIT_products:
    summary = item.summary()
    print(summary["short-name"])

EMITL2ARFL
EMITL1BRAD
EMITL2BCH4PLM
EMITL2BMIN
EMITL1BATT
EMITL2BCH4ENH
EMITL2BCO2ENH
EMITL2BCO2PLM
EMITL3ASA
EMITL4ESM


In [17]:
print(study_period_start.strftime('%Y-%m-%d'),study_period_end.strftime('%Y-%m-%d'))
EMITL1BRAD_products = earthaccess.search_data(
         short_name="EMITL1BRAD",
         bounding_box=domain_box, #box_around_CERES_pixel,
         temporal = (study_period_start.strftime('%Y-%m-%d'),study_period_end.strftime('%Y-%m-%d')),
        )
print('EMITL1BRAD_products',len(EMITL1BRAD_products))

2023-06-01 2023-08-31
EMITL1BRAD_products 5835


### search for MODIS MAIAC data

In [12]:
MAIAC_aerosol_products = earthaccess.search_data(
     short_name="MCD19A2",
     bounding_box=domain_box,
     temporal = (formatted_start,formatted_end)
    )
print('MCD19A2 found',len(MAIAC_aerosol_products))

MAIAC_surf_products = earthaccess.search_data(
     short_name="MCD19A1",
     bounding_box=domain_box,
     temporal = (formatted_start,formatted_end)
    )
print('MCD19A1 found',len(MAIAC_surf_products))

# for item in MAIAC_surf_products:
#     summary = item.summary()
#     print(summary["short-name"])

MCD19A2 found 2278
MCD19A1 found 2277


In [13]:
MAIAC_aerosol_products_list = earthaccess.open(MAIAC_aerosol_products)
MAIAC_surf_products_list = earthaccess.open(MAIAC_surf_products)

QUEUEING TASKS | : 100%|██████████| 2278/2278 [00:00<00:00, 40003.45it/s]
PROCESSING TASKS | : 100%|██████████| 2278/2278 [00:05<00:00, 434.54it/s]
COLLECTING RESULTS | : 100%|██████████| 2278/2278 [00:00<00:00, 636042.11it/s]
QUEUEING TASKS | : 100%|██████████| 2277/2277 [00:00<00:00, 66522.00it/s]
PROCESSING TASKS | : 100%|██████████| 2277/2277 [00:04<00:00, 464.31it/s]
COLLECTING RESULTS | : 100%|██████████| 2277/2277 [00:00<00:00, 533215.91it/s]


In [14]:
MAIAC_aerosol_products_filename = []
for i in range(len(MAIAC_aerosol_products_list)):
    MAIAC_aerosol_products_filename.append(MAIAC_aerosol_products_list[i].full_name.split('/')[-1])
MAIAC_surf_products_filename = []
for i in range(len(MAIAC_surf_products_list)):
    MAIAC_surf_products_filename.append(MAIAC_surf_products_list[i].full_name.split('/')[-1])
    


In [16]:
earthaccess.download(MAIAC_aerosol_products[0], "./MAIAC_data")

[PosixPath('MAIAC_data/MCD19A2.A2023152.h23v05.061.2023153195919.hdf')]

## First step collocation of CERES, EMIT and MAIAC data day by day

In [30]:

lpdaac_creds = get_NASA_creds('lpdaac')
fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=lpdaac_creds['accessKeyId'], 
                          secret=lpdaac_creds['secretAccessKey'], 
                          token=lpdaac_creds['sessionToken'])


current_date = datetime.date(2023,6,2)
while current_date <= study_period_end:
    EMIT_CERES_match_file_allgranule=[]
    print('collocating',current_date)
    print('first search how many EMIT files in the study domain on this day')
    EMITL1BRAD_products = earthaccess.search_data(
         short_name="EMITL1BRAD",
         bounding_box=domain_box, #box_around_CERES_pixel,
         temporal = (current_date.strftime('%Y-%m-%d'),current_date.strftime('%Y-%m-%d')),
        )
    EMITL2ARFL_product = earthaccess.search_data(
         short_name="EMITL2ARFL",
         bounding_box=domain_box, #box_around_CERES_pixel,
         temporal = (current_date.strftime('%Y-%m-%d'),current_date.strftime('%Y-%m-%d')),
        )
    
    print('number of EMIT granules found', len(EMITL1BRAD_products),len(EMITL2ARFL_product))
    n_EMIT_granules = len(EMITL1BRAD_products)
    
    if n_EMIT_granules ==0:
        current_date += datetime.timedelta(days=1)
        continue
    
    EMIT_CERES_match_file={}
    for iEMIT in range(n_EMIT_granules):
        EMIT_lat = EMITL1BRAD_products[iEMIT]['umm']['SpatialExtent']['HorizontalSpatialDomain']['Geometry']['GPolygons'][0]['Boundary']['Points'][2]['Latitude']
        EMIT_lon = EMITL1BRAD_products[iEMIT]['umm']['SpatialExtent']['HorizontalSpatialDomain']['Geometry']['GPolygons'][0]['Boundary']['Points'][2]['Longitude']
        
        for j in range(2):
            filename = EMITL1BRAD_products[iEMIT].data_links()[j].split("/")[-1]
            if "EMIT_L1B_RAD_" in filename:
                L1B_RAD_file= re.sub(r'^https://[^/]+/', 's3://', EMITL1BRAD_products[iEMIT].data_links()[j])
            if "EMIT_L1B_OBS_" in filename:
                L1B_OBS_file = re.sub(r'^https://[^/]+/', 's3://', EMITL1BRAD_products[iEMIT].data_links()[j])
            
        for j in range(3):
            filename = EMITL2ARFL_product[iEMIT].data_links()[j].split("/")[-1]
            if "EMIT_L2A_RFL_001_" in filename:
                L2A_RFL_file = re.sub(r'^https://[^/]+/', 's3://', EMITL2ARFL_product[iEMIT].data_links()[j])
            if "EMIT_L2A_MASK_001_" in filename:
                L2A_MASK_file = re.sub(r'^https://[^/]+/', 's3://', EMITL2ARFL_product[iEMIT].data_links()[j])
    
        

        #print('search for CERES data for each EMIT L1B granule')
        SSF_Aqua_FM3_products = earthaccess.search_data(
         short_name="CER_SSF_Aqua-FM3-MODIS",
         bounding_box=(EMIT_lon-0.5,EMIT_lat-0.5,EMIT_lon+0.5,EMIT_lat+0.5),
         temporal = (current_date.strftime('%Y-%m-%d'),current_date.strftime('%Y-%m-%d'))
        )
        
        MAIAC_aerosol_products = earthaccess.search_data(
            short_name="MCD19A2",
            bounding_box=(EMIT_lon-0.5,EMIT_lat-0.5,EMIT_lon+0.5,EMIT_lat+0.5),
            temporal = (current_date.strftime('%Y-%m-%d'),current_date.strftime('%Y-%m-%d'))
            )
        # print('number of MAIAC aerosol product',len(MAIAC_aerosol_products))
        # print(MAIAC_aerosol_products[0].data_links())

        MAIAC_surf_products = earthaccess.search_data(
            short_name="MCD19A1",
            bounding_box=(EMIT_lon-0.5,EMIT_lat-0.5,EMIT_lon+0.5,EMIT_lat+0.5),
            temporal = (current_date.strftime('%Y-%m-%d'),current_date.strftime('%Y-%m-%d'))
            )
        
        
        # print('number of MAIAC surface product',len(MAIAC_surf_products))
        # print(MAIAC_surf_products[0].data_links())

        #print('number of CERES files found for this EMIT granule',len(SSF_Aqua_FM3_products))
        #for jCERES in range(len(SSF_Aqua_FM3_products)):

        if (len(SSF_Aqua_FM3_products) ==0) or (len(MAIAC_aerosol_products)==0): continue
            
        CERES_filename = [SSF_Aqua_FM3_products[jCERES].data_links()[0] for jCERES in range(len(SSF_Aqua_FM3_products))]
        MAIAC_surf_filename     = [MAIAC_surf_products[kMAIAC].data_links()[0] for kMAIAC in range(len(MAIAC_surf_products))]
        MAIAC_aerosol_filename = [MAIAC_aerosol_products[kMAIAC].data_links()[0] for kMAIAC in range(len(MAIAC_aerosol_products))]
        
        EMIT_CERES_match_file['EMIT_files']= [L1B_RAD_file,L1B_OBS_file,L2A_RFL_file,L2A_MASK_file]
        EMIT_CERES_match_file['EMIT_product_types']= ['RAD','OBS','RFL','MASK']
        EMIT_CERES_match_file['CERES_files'] = CERES_filename
        EMIT_CERES_match_file['MAIAC_surf_files'] = MAIAC_surf_filename
        EMIT_CERES_match_file['MAIAC_aerosol_files'] = MAIAC_aerosol_filename
        EMIT_CERES_match_file_allgranule.append(EMIT_CERES_match_file)
        #print(EMIT_CERES_match_file_allgranule)
    
        pickle_filename = 'EMIT_CERES_MAIAC_match_file_'+current_date.strftime('%Y-%m-%d')+'_EMIT_granule_{0:03d}.pickle'.format(iEMIT)
        #print('saving',pickle_filename)

        directory = "./EMIT_CERES_MAIAC_match_files"
        os.makedirs(directory, exist_ok=True)                # ensure folder exists

        filepath = os.path.join(directory, pickle_filename)

        with open(filepath, "wb") as f:
            pickle.dump(EMIT_CERES_match_file, f, protocol=pickle.HIGHEST_PROTOCOL)

    current_date += datetime.timedelta(days=1)
    
# with open('EMIT_CERES_match_file_alldays.pickle', 'rb') as handle:
#     b = pickle.load(handle)
            
# pickle_filename = 'EMIT_CERES_MAIAC_match_file_'+current_date.strftime('%Y-%m-%d')+'.pickle'
# directory = "./EMIT_CERES_MAIAC_match_files"
# os.makedirs(directory, exist_ok=True)
# filepath = os.path.join(directory, pickle_filename)
# with open(filepath, 'wb') as handle:
#     pickle.dump(EMIT_CERES_match_file_allgranule,handle, protocol=pickle.HIGHEST_PROTOCOL)
    

collocating 2023-06-02
first search how many EMIT files in the study domain on this day
number of EMIT granules found 102 102
collocating 2023-06-03
first search how many EMIT files in the study domain on this day
number of EMIT granules found 92 92
collocating 2023-06-04
first search how many EMIT files in the study domain on this day
number of EMIT granules found 96 96
collocating 2023-06-05
first search how many EMIT files in the study domain on this day
number of EMIT granules found 114 114
collocating 2023-06-06
first search how many EMIT files in the study domain on this day
number of EMIT granules found 94 94
collocating 2023-06-07
first search how many EMIT files in the study domain on this day
number of EMIT granules found 96 96
collocating 2023-06-08
first search how many EMIT files in the study domain on this day
number of EMIT granules found 87 87
collocating 2023-06-09
first search how many EMIT files in the study domain on this day
number of EMIT granules found 58 58
coll

In [32]:
with open('./EMIT_CERES_MAIAC_match_files/EMIT_CERES_MAIAC_match_file_2023-08-01_EMIT_granule_027.pickle', 'rb') as file:
        collocation_data_dict = pickle.load(file)
print(collocation_data_dict['EMIT_files'][0])


s3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230801T091247_2321306_029/EMIT_L1B_RAD_001_20230801T091247_2321306_029.nc


In [33]:
with open('./EMIT_CERES_MAIAC_match_files/EMIT_CERES_MAIAC_match_file_2023-08-01_EMIT_granule_021.pickle', 'rb') as file:
        collocation_data_dict = pickle.load(file)
print(collocation_data_dict['EMIT_files'][0])

s3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230801T091035_2321306_023/EMIT_L1B_RAD_001_20230801T091035_2321306_023.nc


## go through collocation data and collocated EMIT and MAIAC data

In [18]:
# routine needed to compute the distance of EMIT or MAIAC pixels to the target CERES pixel
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Compute the great circle distance between two points
    on the earth (specified in decimal degrees) using the haversine formula.
    All inputs can be scalars or numpy arrays.
    """
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # haversine formula
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Earth radius in kilometers (mean radius ~6371 km)
    km = 6371 * c
    return km

In [19]:
CERES_clearsky_fraction_threshold = 100.0 # the threshold for selecting clear-sky CERES SSF FOV
EMIT_CERES_distance_threshold = 20.0
pickle_directory = "./EMIT_CERES_MAIAC_match_files/"
        

#loop through the dates        
current_date = datetime.date(2023,6,1)
while current_date <= study_period_end:
    # get auth for AWS for each day
    lpdaac_creds = get_NASA_creds('lpdaac')
    fs_s3 = s3fs.S3FileSystem(anon=False, 
                          key=lpdaac_creds['accessKeyId'], 
                          secret=lpdaac_creds['secretAccessKey'], 
                          token=lpdaac_creds['sessionToken'])
    
    pickle_filename_prefix = 'EMIT_CERES_MAIAC_match_file_'+current_date.strftime('%Y-%m-%d')+'_EMIT_granule_*.pickle'
    #filepath = os.path.join(pickle_directory, pickle_filename)
    # load all the collocation files for each day
    collocation_files = glob.glob(pickle_directory+pickle_filename_prefix)
    
    if len(collocation_files)==0:
        current_date += datetime.timedelta(days=1)
        continue
   
    N_EMIT_granule = len(collocation_files)
    print('date',current_date,'number of EMIT granules',N_EMIT_granule)
    # for col_file in collocation_files:
    #     with open(col_file, 'rb') as file:
    #         collocation_data_dict = pickle.load(file)
    

    for iEMIT in range(N_EMIT_granule):
        print('processing granule #',iEMIT)
        pickle_filename = pickle_directory+'EMIT_CERES_MAIAC_match_file_'+current_date.strftime('%Y-%m-%d')+'_EMIT_granule_{0:03d}.pickle'.format(iEMIT)
        
        with open(pickle_filename, 'rb') as file:
            collocation_data_dict = pickle.load(file)
        
        print('read in EMIT data from AWS')
        #print(collocation_data_dict['EMIT_files'])
        EMIT_granule_data = EMIT_lib.read_single_EMIT_granule_data_aws(collocation_data_dict['EMIT_files'],
                                                              collocation_data_dict['EMIT_product_types'],fs_s3)
        # find the boundaries of the EMIT granule
        EMIT_granule_lon_min = EMIT_granule_data['OBS_location'].lon.values.min()
        EMIT_granule_lon_max = EMIT_granule_data['OBS_location'].lon.values.max()
        EMIT_granule_lat_min = EMIT_granule_data['OBS_location'].lat.values.min()
        EMIT_granule_lat_max = EMIT_granule_data['OBS_location'].lat.values.max()
        
        
        print('read in CERES data from local directory')
        print('number of CERES files',len(collocation_data_dict['CERES_files']))
        for jCERES in range(len(collocation_data_dict['CERES_files'])):
            filename = collocation_data_dict['CERES_files'][jCERES].split("/")[-1]
            CERES_data = xr.open_dataset('./CERES_SSF/'+filename,engine='netcdf4')
            print('processing CERES granule #',jCERES,filename)
            # let's see if there are any CERES pixels in the EMIT granule
            CERES_lon = CERES_data['Longitude of CERES FOV at surface']
            idx = CERES_lon.values >180.0
            CERES_lon[idx] = CERES_lon[idx] - 360.0 # convert the range from 0~360 to -180~180
            CERES_lat = 90.0-CERES_data['Colatitude of CERES FOV at surface']
            CERES_in_domain = (CERES_lon.values<=EMIT_granule_lon_max) & (CERES_lon.values>=EMIT_granule_lon_min)  & \
                              (CERES_lat.values<=EMIT_granule_lat_max) & (CERES_lat.values>=EMIT_granule_lat_min)
            print('number of CERES pixels in the EMIT granule', np.sum(CERES_in_domain))
            daytime_pixels  = CERES_data['CERES solar zenith at surface'].values<90.0
            print('number of daytime pixel',np.sum(daytime_pixels))
            # seect pixels with the clear-sky fraction larger than the threshold
            clearsky_pixels = CERES_data['Clear/layer/overlap percent coverages'][:,0].values >= CERES_clearsky_fraction_threshold
            print('number of clearsky pixels ',np.sum(clearsky_pixels))
            selected_CERES_pixels = CERES_in_domain & daytime_pixels & clearsky_pixels
            print('cloud fractions of selected CERES granule',CERES_data['Clear/layer/overlap percent coverages'][:,0].values[selected_CERES_pixels])
            print('number of daytime & clear-sky CERES pixels in the EMIT granule', np.sum(selected_CERES_pixels))

            if np.sum(selected_CERES_pixels) ==0:
                # if there are no CERES pixels go to next EMIT granule
                continue
            print('go through selected the CERES pixels' ,np.sum(selected_CERES_pixels))
            N_pixels=np.sum(selected_CERES_pixels)
            for kpixel in range(N_pixels):
                print('lat lon of CERES pixel',kpixel,
                      CERES_lat[selected_CERES_pixels].values[kpixel],
                      CERES_lon[selected_CERES_pixels].values[kpixel])
                
                # find EMIT pixels with in the CERES pixel
                EMIT_to_CERES_distance = haversine_np(CERES_lon[selected_CERES_pixels].values[kpixel],
                                                      CERES_lat[selected_CERES_pixels].values[kpixel],
                                                      EMIT_granule_data['OBS_location'].lon.values.flatten(),
                                                      EMIT_granule_data['OBS_location'].lat.values.flatten())
                EMIT_pixels_in_CERES = EMIT_to_CERES_distance < EMIT_CERES_distance_threshold
                # print(mask)
                # selected_indices = np.where(mask)[0]
                # print(EMIT_granule_data['OBS_location'].lon.values.shape)
                print('numner of EMIT pxiels in the CERES pixel',np.sum(EMIT_pixels_in_CERES),'out of',EMIT_granule_data['OBS_location'].lon.size)

                #------------- get the EMIT radiance data here -------------------------------#
                
                #process the MAIAC data 
                MAIAC_aerosol_product_index = MAIAC_aerosol_products_filename.index(collocation_data_dict['MAIAC_aerosol_files'][0].split('/')[-1])
                MAIAC_surf_product_index = MAIAC_surf_products_filename.index(collocation_data_dict['MAIAC_surf_files'][0].split('/')[-1])
                print(MAIAC_aerosol_products_filename[MAIAC_aerosol_product_index]==
                     collocation_data_dict['MAIAC_aerosol_files'][0].split('/')[-1])
                print(MAIAC_surf_products_filename[MAIAC_surf_product_index]==
                      collocation_data_dict['MAIAC_surf_files'][0].split('/')[-1])
                earthaccess.download(MAIAC_aerosol_products[MAIAC_aerosol_product_index], "./MAIAC_data")
                earthaccess.download(MAIAC_surf_product[MAIAC_surf_product_index], "./MAIAC_data")
            
    
    current_date += datetime.timedelta(days=1)
                
        
    

    
                


# while current_date <= study_period_end:
#     with open(pickle_filename, 'wb') as handle:
#         pickle.dump(EMIT_CERES_math_file_allgranule, handle, protocol=pickle.HIGHEST_PROTOCOL)
        
#     current_date += datetime.timedelta(days=1)

date 2023-06-01 number of EMIT granules 104
processing granule # 0
read in EMIT data from AWS
read in CERES data from local directory
number of CERES files 2
processing CERES granule # 0 CER_SSF_Aqua-FM3-MODIS_Edition4A_407405.2023060110.hdf
number of CERES pixels in the EMIT granule 20
number of daytime pixel 49456
number of clearsky pixels  8409
cloud fractions of selected CERES granule [100. 100. 100.]
number of daytime & clear-sky CERES pixels in the EMIT granule 3
go through selected the CERES pixels 3
lat lon of CERES pixel 0 14.029312 50.347298
numner of EMIT pxiels in the CERES pixel 26734 out of 1589760
True
True


NameError: name 'MAIAC_surf_product' is not defined

In [37]:
earthaccess.download(MAIAC_aerosol_products_list[MAIAC_aerosol_product_index])

DispatchError: ('_get: 0 methods found', (<class 'earthaccess.store.Store'>, <class 'earthaccess.store.EarthAccessFile'>, <class 'pathlib.PosixPath'>), [])

In [14]:
collocation_data_dict['MAIAC_aerosol_files']

['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/MCD19A2.061/MCD19A2.A2023152.h23v07.061.2023153185819/MCD19A2.A2023152.h23v07.061.2023153185819.hdf',
 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/MCD19A2.061/MCD19A2.A2023152.h22v07.061.2023153185850/MCD19A2.A2023152.h22v07.061.2023153185850.hdf']

In [14]:
with open('EMIT_CERES_math_file_2023-06-06.pickle', 'rb') as file:
        collocation_data_dict = pickle.load(file)

In [16]:
collocation_data_dict[4]['EMIT_files']

['s3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230606T101633_2315707_044/EMIT_L1B_RAD_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230606T101633_2315707_044/EMIT_L1B_OBS_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230606T101633_2315707_044/EMIT_L2A_RFL_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230606T101633_2315707_044/EMIT_L2A_MASK_001_20230606T101633_2315707_044.nc']

In [17]:
collocation_data_dict[10]['EMIT_files']

['s3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230606T101633_2315707_044/EMIT_L1B_RAD_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL1BRAD.001/EMIT_L1B_RAD_001_20230606T101633_2315707_044/EMIT_L1B_OBS_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230606T101633_2315707_044/EMIT_L2A_RFL_001_20230606T101633_2315707_044.nc',
 's3://lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230606T101633_2315707_044/EMIT_L2A_MASK_001_20230606T101633_2315707_044.nc']

### go through the CERES SSF data and collocate with EMIT and MODIS

In [21]:
CRF_SSF_Aqua_FM3_filelist = glob.glob('./CERES_SSF/*.hdf')
print(len(CRF_SSF_Aqua_FM3_filelist))

1229


In [25]:
CERES_clearsky_fraction_threshold = 100.0 # the threshold for selecting clear-sky CERES SSF FOV
plot_on_map=True
for fn in (CRF_SSF_Aqua_FM3_filelist[0:5]):
    print(fn)
    date_str = fn.split('.')[2]

    # Convert the string to a datetime object using the appropriate format:
    # %Y -> year, %m -> month, %d -> day, %H -> hour.
    CERES_time = datetime.datetime.strptime(date_str, "%Y%m%d%H")
    
    print(CERES_time)  # Output: 2023-06-01 04:00:00
    # read in the CERES products
    SSF_product = xr.open_dataset(fn,engine='netcdf4')
    # find all the pixels within the study domain
    CERES_lon = SSF_product['Longitude of CERES FOV at surface']
    idx = CERES_lon.values >180.0
    CERES_lon[idx] = CERES_lon[idx] - 360.0 # convert the range from 0~360 to -180~180
    CERES_lat = 90.0-SSF_product['Colatitude of CERES FOV at surface']
    CERES_in_domain = (CERES_lon.values<=domain_lon_max) & (CERES_lon.values>=domain_lon_min)  & \
                      (CERES_lat.values<=domain_lat_max) & (CERES_lat.values>=domain_lat_min) 
    # select pixels with solar zenith angle smaller than 90 degree
    daytime_pixels  = SSF_product['CERES solar zenith at surface'].values<90.0
    # seect pixels with the clear-sky fraction larger than the threshold
    clearsky_pixels = SSF_product['Clear/layer/overlap percent coverages'][:,0].values >= CERES_clearsky_fraction_threshold
    selected_CERES_pixels = CERES_in_domain & daytime_pixels & clearsky_pixels

    # if no pixels satisfied the selection conditions, goto next file
    if selected_CERES_pixels.sum()<=0:
        continue

    npixels = np.sum(selected_CERES_pixels)

    CERES_time_start = CERES_time.strftime('%Y-%m-%d')
    CERES_time_end  = CERES_time.strftime('%Y-%m-%d')
    EMITL1BRAD_products = earthaccess.search_data(
         short_name="EMITL1BRAD",
         bounding_box=domain_box,
         temporal = (CERES_time_start,CERES_time_end),
        study_period_start.strftime('%Y-%m-%d')
        )
    print('total number of EMIT L1',len(EMITL1BRAD_products))

    print('total number of selected CERES pixels', npixels)
    
    print(formatted_start,formatted_end)
    for ipixel in range(npixels):
        box_around_CERES_pixel = (  CERES_lon[selected_CERES_pixels][ipixel] - 10.1, CERES_lat[selected_CERES_pixels][ipixel] - 10.1,
                                    CERES_lon[selected_CERES_pixels][ipixel] - 10.1, CERES_lat[selected_CERES_pixels][ipixel] - 10.1 )
        
        
        # print(formatted_start)  # Output: 2023-06-01
        # print(formatted_end)    # Output: 2023-08-31
        
    
    
    if plot_on_map:
        # Create a figure with a PlateCarree projection (suitable for geographic data)
        fig = plt.figure(figsize=(12, 6))
        ax = plt.axes(projection=ccrs.PlateCarree())
        
        # Set the view to display the entire globe
        ax.set_global()
        
        # Add coastlines for context
        ax.coastlines()
        
        # Plot the points as red stars
        ax.plot(CERES_lon[selected_CERES_pixels], CERES_lat[selected_CERES_pixels], 
                marker='o', color='red', linestyle='',
                markersize=10, transform=ccrs.PlateCarree())
        
        plt.title("CERES Points on a Global Map")
        plt.show()

SyntaxError: positional argument follows keyword argument (1103913813.py, line 40)

### for each CERES pixel in the domain find collocated EMIT and MODIS data

In [14]:
for k in SSF_product.keys():
    print(k)

Time of observation
Radius of satellite from center of Earth at observation
X component of satellite inertial velocity
Y component of satellite inertial velocity
Z component of satellite inertial velocity
Colatitude of subsatellite point at surface at observation
Longitude of subsatellite point at surface at observation
Colatitude of subsolar point at surface at observation
Longitude of subsolar point at surface at observation
Colatitude of CERES FOV at surface
Longitude of CERES FOV at surface
Scan sample number
Packet number
Cone angle of CERES FOV at satellite
Clock angle of CERES FOV at satellite wrt inertial velocity
Rate of change of cone angle
Rate of change of clock angle
Along-track angle of CERES FOV at surface
Cross-track angle of CERES FOV at surface
CERES viewing zenith at surface
CERES solar zenith at surface
CERES relative azimuth at surface
CERES viewing azimuth at surface wrt North
Altitude of surface above sea level
Surface type index
Surface type percent coverage
CER

In [20]:
SSF_product['Clear/layer/overlap percent coverages'][:,0]


<xarray.DataArray 'Clear/layer/overlap percent coverages' (
                                                           Footprints (FOV): 98798)> Size: 395kB
[98798 values with dtype=float32]
Dimensions without coordinates: Footprints (FOV)
Attributes:
    units:        N/A
    format:       F18.9
    valid_range:  [  0. 100.]